# Supplementary Figure S2 — dyngen with three sampling times fails (S21 vs S21b)

Sources: `benchmarks/dyngen/s21/scores.csv` (T = 3) and `benchmarks/dyngen/s21b/scores.csv` (T = 8, same
simulations re-sampled). Same columns (RESULT_S21b.md).

In [1]:
from _style import *

prov = Provenance("FigS2", notebook="notebooks/supp_s21_three_timepoints.ipynb", kind="supplementary")
t3 = pd.read_csv(prov.use("benchmarks/dyngen/s21/scores.csv", "a,b,c", "S21"))
t8 = pd.read_csv(prov.use("benchmarks/dyngen/s21b/scores.csv", "a,b,c", "S21b"))
t8["dataset"] = t8.dataset.str.replace("_T8", "", regex=False)
V = prov.value
fig, axs = plt.subplots(1, 3, figsize=(DOUBLE_COL, 2.4), gridspec_kw={"wspace": 0.62})
C3, C8 = C_BASE, C_FLOW

# a — region sizes
ax = axs[0]
bif = sorted(t3[t3.backbone == "bifurcating"].dataset.unique())
for i, ds in enumerate(bif):
    for T, d, col, dx in [(3, t3, C3, -0.2), (8, t8, C8, 0.2)]:
        r = d[(d.dataset == ds) & (d.operator == "flow")].iloc[0]
        ax.bar(i + dx - 0.09, r.n_fork, width=0.17, color=col)
        ax.bar(i + dx + 0.09, r.n_early, width=0.17, color=col, alpha=0.45)
        V(f"a_T{T}_{ds}_n_fork", int(r.n_fork)); V(f"a_T{T}_{ds}_n_early", int(r.n_early)); V(f"a_T{T}_{ds}_n_committed", int(r.n_committed))
ax.set_xticks(range(len(bif))); ax.set_xticklabels([b.replace("bifurcating_v2_", "") for b in bif])
ax.set_ylabel("cells"); ax.set_title("fork (solid) and\nearly (light) cells", fontsize=6.3)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C3, label="T = 3 (S21)"), plt.Rectangle((0, 0), 1, 1, color=C8, label="T = 8 (S21b)")], fontsize=6, loc="upper right", borderaxespad=0.1)
ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
panel_letter(ax, "a", x=-0.35)
prov.panel("a", "Cells in the fork and early (pre-branch) truth regions at 3 vs 8 sampling times")

# b — rotation
ax = axs[1]
for j, (T, d, col) in enumerate([(3, t3, C3), (8, t8, C8)]):
    f = d[d.operator == "flow"]
    for i, p in enumerate(["s0", "s1", "s2"]):
        c = f[f.dataset == f"cycle_{p}"].sort_values("drift_seed").median_rotation.values
        l = f[f.dataset == f"linear_{p}"].sort_values("drift_seed").median_rotation.values
        x = i * 2.5 + j
        ax.scatter(np.full(3, x - 0.15), c, color=col, s=10, zorder=3)
        ax.scatter(np.full(3, x + 0.15), l, facecolor="white", edgecolor=col, s=10, zorder=3)
        V(f"b_T{T}_{p}_cycle_gt_linear_all_seeds", bool(np.all(c > l)))
        V(f"b_T{T}_{p}_cycle_median_rotation_max", float(c.max())); V(f"b_T{T}_{p}_linear_median_rotation_max", float(l.max()))
ax.set_xticks([0.5, 3.0, 5.5]); ax.set_xticklabels(["s0", "s1", "s2"]); ax.set_ylabel("median rotational index")
ax.set_title("cycle (●) vs linear (○)\ngrey T = 3, blue T = 8", fontsize=6.3)
panel_letter(ax, "b", x=-0.35)
prov.panel("b", "Median rotational index, cycle vs linear, at T=3 and T=8 (three drift seeds)")

# c — fraction Re λ1 > 0 in early (pre-branch) cells
ax = axs[2]
for j, (T, d, col) in enumerate([(3, t3, C3), (8, t8, C8)]):
    f = d[(d.operator == "flow") & (d.backbone == "bifurcating")]
    for i, ds in enumerate(bif):
        v = f[f.dataset == ds].fraction_positive_early
        ax.scatter(i + (j - 0.5) * 0.3 + np.array([-0.05, 0, 0.05]), v, color=col, s=10, zorder=3)
        V(f"c_T{T}_{ds}_frac_pos_early_min", float(v.min())); V(f"c_T{T}_{ds}_frac_pos_early_max", float(v.max()))
ax.set_xticks(range(len(bif))); ax.set_xticklabels([b.replace("bifurcating_v2_", "") for b in bif])
ax.set_ylabel("fraction Re λ₁ > 0, early cells"); ax.set_ylim(-0.05, 1.05)
ax.set_title("pre-branch instability\n(T = 3 artefact)", fontsize=6.3)
panel_letter(ax, "c", x=-0.35)
prov.panel("c", "Fraction of early (pre-branch) cells with Re lambda1 > 0 at T=3 vs T=8")
save(fig, "FigS2", kind="supplementary")

[save] supplementary/FigS2: 158.8 x 60.6 mm, PNG 3752 x 1431 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS2.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS2.png'))

In [2]:
v = prov.values
b = bif
prov.legend(f"""
**Supplementary Figure S2 | Three sampling times are too coarse on dyngen (S21, pre-registered; both gates failed or not evaluable) vs eight (S21b).**
Same simulations, synchronised sampling at T = 3 (S21) or T = 8 (S21b); flow J drift seeds 0–2. **a**, Cells in the fork region at T = 3:
{v[f'a_T3_{b[0]}_n_fork']}, {v[f'a_T3_{b[1]}_n_fork']}, {v[f'a_T3_{b[2]}_n_fork']} (s10, s16, s18; F1 not evaluable for s16 and s18) vs
{v[f'a_T8_{b[0]}_n_fork']}, {v[f'a_T8_{b[1]}_n_fork']}, {v[f'a_T8_{b[2]}_n_fork']} at T = 8. **b**, Rotational index, cycle vs linear: cycle above linear on all
drift seeds at T = 3 for s0/s1/s2: {', '.join(str(v[f'b_T3_{p}_cycle_gt_linear_all_seeds']) for p in ['s0', 's1', 's2'])}; at T = 8:
{', '.join(str(v[f'b_T8_{p}_cycle_gt_linear_all_seeds']) for p in ['s0', 's1', 's2'])}. **c**, Fraction of early (pre-branch) cells with Re λ₁ > 0:
{rng_str(min(v[f'c_T3_{d}_frac_pos_early_min'] for d in b), max(v[f'c_T3_{d}_frac_pos_early_max'] for d in b))} at T = 3 on every dataset and seed; at T = 8,
s10 {rng_str(v[f'c_T8_{b[0]}_frac_pos_early_min'], v[f'c_T8_{b[0]}_frac_pos_early_max'])}, s16 {rng_str(v[f'c_T8_{b[1]}_frac_pos_early_min'], v[f'c_T8_{b[1]}_frac_pos_early_max'])},
s18 {rng_str(v[f'c_T8_{b[2]}_frac_pos_early_min'], v[f'c_T8_{b[2]}_frac_pos_early_max'])}; the S21 reading "pre-branch λ₁ > 0" was withdrawn as a sampling artefact.
""")
prov.write()